## pickle → SCIDAC converter

Converts qlat topological charge density fields (stored as numpy arrays in a pickle)
into SCIDAC/LIME files readable by Grid's `ScidacReader`.

**Axis convention**:  
qlat stores fields in `(x, y, z, t)` order (x slowest, t fastest in C-order memory).  
Grid's lexicographic order has x fastest, t slowest.  
The conversion is `.T` (reverses all 4 axes), then C-order flatten gives Grid binary order.

**Record structure** (matches `ScidacWriter::writeScidacFieldRecord`):
1. `grid-format` (MB=1,ME=0) — FieldMetaData XML  
2. `scidac-record-xml` (MB=0,ME=0) — emptyUserRecord XML  
3. `scidac-private-record-xml` (MB=0,ME=0) — scidacRecord XML  
4. `ildg-binary-data` (MB=0,ME=0) — big-endian complex128 payload  
5. `scidac-checksum` (MB=0,ME=1) — SciDAC checksum XML  

**Source**: `32c-hmc-test-gen.ipynb` reads `f_tadpole_loop.field` from qlat  
(`rand_vol_u1_idx-0` and `-1`: two stochastic U(1) estimates of the DWF midpoint TCD, i.e. `q_A`).  
Config: trajectory 702, 32⁴ lattice.

In [ ]:
import struct
import pickle
import numpy as np
import os

In [ ]:
import struct, zlib, pickle
import numpy as np
import os

# ── LIME record writer ────────────────────────────────────────────────────────
LIME_MAGIC   = 0x456789AB01234567
LIME_VERSION = 1

def _lime_record(type_str: str, data: bytes, MB: bool, ME: bool) -> bytes:
    """Pack one LIME record (148-byte header + data padded to 8-byte boundary)."""
    flags  = (0x8000 if MB else 0) | (0x4000 if ME else 0)
    type_b = type_str.encode('ascii').ljust(128, b'\x00')[:128]
    header = struct.pack('>QHHQ128s', LIME_MAGIC, LIME_VERSION, flags, len(data), type_b)
    pad    = (8 - len(data) % 8) % 8
    return header + data + b'\x00' * pad

# ── SciDAC checksum (mirrors BinaryIO.h::ScidacChecksum) ─────────────────────
def _rotl32(x: int, n: int) -> int:
    n &= 31
    return ((x << n) | (x >> (32 - n))) & 0xFFFFFFFF if n else (x & 0xFFFFFFFF)

def _scidac_checksums(payload: bytes, Nsites: int):
    """
    Grid computes ScidacChecksum *after* htobe (write) / *before* be64toh (read),
    so CRC is taken over the big-endian bytes on disk.
    """
    mv = memoryview(payload)
    csuma = csumb = 0
    for site in range(Nsites):
        crc   = zlib.crc32(mv[site*16 : site*16+16]) & 0xFFFFFFFF
        csuma ^= _rotl32(crc, site % 29)
        csumb ^= _rotl32(crc, site % 31)
    return csuma, csumb

In [ ]:
# ── Grid-compatible SCIDAC writer for a complex scalar field ──────────────────
def write_scidac_complex(fname: str, field: np.ndarray):
    """
    Write a 4D complex scalar field to a Grid-compatible SCIDAC/LIME file.

    Parameters
    ----------
    fname : output filename
    field : numpy array, shape (Nt, Nz, Ny, Nx), dtype complex128
            Grid order: t slowest, x fastest (pass arr.T from qlat's (x,y,z,t)).
    """
    assert field.ndim == 4
    Nt, Nz, Ny, Nx = field.shape
    Nsites = Nx * Ny * Nz * Nt

    # Binary payload: big-endian interleaved re/im
    # C-order ravel over (Nt,Nz,Ny,Nx) → t slowest, x fastest = Grid lexicographic order
    flat    = np.ascontiguousarray(field, dtype=np.complex128).ravel()
    be      = np.empty(2 * len(flat), dtype='>f8')
    be[0::2] = flat.real
    be[1::2] = flat.imag
    payload = be.tobytes()

    # SciDAC checksums (computed on big-endian data, matching Grid's write path)
    print(f'  computing SciDAC checksums over {Nsites:,} sites…', end=' ', flush=True)
    csuma, csumb = _scidac_checksums(payload, Nsites)
    print(f'done  csuma=0x{csuma:08x}  csumb=0x{csumb:08x}')

    # Record 1: grid-format — FieldMetaData
    field_meta_xml = (
        '<?xml version="1.0"?>'
        '<FieldMetaData>'
          '<nd>4</nd>'
          f'<dimension><elem>{Nx}</elem><elem>{Ny}</elem><elem>{Nz}</elem><elem>{Nt}</elem></dimension>'
          '<boundary><elem>PERIODIC</elem><elem>PERIODIC</elem><elem>PERIODIC</elem><elem>PERIODIC</elem></boundary>'
          '<data_start>0</data_start><hdr_version></hdr_version><storage_format></storage_format>'
          '<link_trace>0</link_trace><plaquette>0</plaquette><checksum>0</checksum>'
          f'<scidac_checksuma>{csuma}</scidac_checksuma>'
          f'<scidac_checksumb>{csumb}</scidac_checksumb>'
          '<sequence_number>0</sequence_number><data_type>GRID_D_Complex</data_type>'
          '<ensemble_id></ensemble_id><ensemble_label></ensemble_label><ildg_lfn></ildg_lfn>'
          '<creator>pickle_to_scidac.py</creator><creator_hardware></creator_hardware>'
          '<creation_date></creation_date><archive_date></archive_date>'
          '<floating_point>IEEE64BIG</floating_point>'
        '</FieldMetaData>'
    ).encode()

    # Record 2: scidac-record-xml — emptyUserRecord
    user_xml = b'<?xml version="1.0"?><emptyUserRecord><dummy>0</dummy></emptyUserRecord>'

    # Record 3: scidac-private-record-xml — scidacRecord
    # datatype "GRID_D_Complex" matches ScidacRecordTypeString<LatticeComplexD>
    scidac_rec_xml = (
        '<?xml version="1.0"?>'
        '<scidacRecord>'
          '<version>1</version><date></date><recordtype>0</recordtype>'
          '<datatype>GRID_D_Complex</datatype><precision>D</precision>'
          '<colors>1</colors><spins>1</spins><typesize>16</typesize><datacount>1</datacount>'
        '</scidacRecord>'
    ).encode()

    # Record 5: scidac-checksum — hex strings without "0x" prefix
    checksum_xml = (
        '<?xml version="1.0"?>'
        '<scidacChecksum>'
          '<version>1</version>'
          f'<suma>{csuma:x}</suma>'
          f'<sumb>{csumb:x}</sumb>'
        '</scidacChecksum>'
    ).encode()

    with open(fname, 'wb') as f:
        f.write(_lime_record('grid-format',                field_meta_xml, MB=True,  ME=False))
        f.write(_lime_record('scidac-record-xml',          user_xml,       MB=False, ME=False))
        f.write(_lime_record('scidac-private-record-xml',  scidac_rec_xml, MB=False, ME=False))
        f.write(_lime_record('ildg-binary-data',           payload,        MB=False, ME=False))
        f.write(_lime_record('scidac-checksum',            checksum_xml,   MB=False, ME=True))

    print(f'  wrote {fname}  ({os.path.getsize(fname)/1024/1024:.1f} MB)')

In [ ]:
# ── Load pickle ───────────────────────────────────────────────────────────────
pickle_path = '../../data/32c-hmc-test-demo.pickle'   # adjust if needed
out_dir     = 'data'
os.makedirs(out_dir, exist_ok=True)

with open(pickle_path, 'rb') as f:
    data = pickle.load(f)

print('Keys:', list(data.keys()))
for k, v in data.items():
    print(f'  {k}: shape={v.shape}  dtype={v.dtype}')

In [ ]:
# ── Convert and write ─────────────────────────────────────────────────────────
for key, arr in data.items():
    print(f'\n{key}:')
    print(f'  Q (real) = {arr.real.sum():.6f}')
    print(f'  imag rms = {np.sqrt(np.mean(arr.imag**2)):.3e}')

    # qlat: (x,y,z,t) x-slowest  →  Grid: (t,z,y,x) t-slowest, x-fastest
    field_grid = arr.T    # .T reverses all 4 axes

    out_path = os.path.join(out_dir, f'{key}.scidac')
    write_scidac_complex(out_path, field_grid)

In [ ]:
# ── Sanity check: RMS difference between the two stochastic estimates ─────────
# topo_field_0 and topo_field_1 are two independent U(1) source estimates of q_A.
# Their RMS difference / sqrt(2) is the per-estimator stochastic noise.
f0 = data['topo_field_0']
f1 = data['topo_field_1']
rms_diff = np.sqrt(np.sum((f1 - f0).real**2) / 2)
print(f'RMS(field_1 - field_0) / sqrt(2) = {rms_diff:.6f}  (stochastic noise per estimate)')
print(f'Noise / |Q|             = {rms_diff / abs(f0.real.sum()):.4f}')